# StoCFL: efeito da fração de clientes por rodada (m)

Pipeline de teste da **clusterização** do StoCFL (`StoCFL-FCC+CFL.py --train 0`) em Rotated MNIST, variando a fração `m` de clientes sorteados por rodada: **100%, 50%, 25% e 10%**.

- **Cenário:** Rotated MNIST, `n = 200` clientes, `k = 4` grupos reais (0°, 90°, 180°, 270°; o cliente `c` pertence ao grupo `c * k // n`).
- **Sem treino:** só a observação de `Ψ` e a fusão de clusters. O que muda com `m` é quantos clientes são observados e quantos clusters entram na matriz de cosseno a cada rodada.
- **Rodadas:** no máximo 50.
- **Como roda:** cada `m` chama o script por `subprocess`, sem alterar o código do StoCFL. O notebook lê o `cluster.pkl` e o log de cada execução.
- **Ambiente:** o kernel é o `.venv` da raiz (ver o README da raiz). Na primeira execução, o MNIST é baixado e as partições rotacionadas são gravadas em `datasets/rotated_mnist_200_<seed>/`.

Métricas por rodada, contra o grupo real de cada cliente:

| Métrica | Significado |
|---|---|
| nº de clusters | tende a `k = 4` quando `τ` está bem calibrado |
| ARI | 1 = partição igual à real |
| pureza | fração de clientes cujo cluster é dominado pelo grupo deles |
| cobertura | fração dos clientes já sorteados (e, portanto, com `Ψ`) |

In [ ]:
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from sklearn.metrics import adjusted_rand_score

PROJETO = Path.cwd()
SCRIPT = "StoCFL-FCC+CFL.py"
assert (PROJETO / SCRIPT).exists(), "abra o notebook a partir de projetos/StoCFL/"

# Parâmetros do teste
N = 200                           # nº de clientes (divisível por K)
K = 4                             # grupos reais do Rotated MNIST: 0°, 90°, 180°, 270°
ROUNDS = 50                       # nº de rodadas de comunicação (no máximo 50)
TAU = 0.9                         # limiar de cosseno para fundir clusters (--tau)
MU = 0.05                         # só entra no nome da pasta de logs: sem treino, não afeta a clusterização
SEEDS = [0]                       # cada seed gera uma nova partição dos dados (~750 MB em disco)
FRACOES = [1.0, 0.5, 0.25, 0.1, 0.05]   # m: fração dos clientes sorteada por rodada
REEXECUTAR = False                # True ignora execuções já concluídas e roda de novo

assert ROUNDS <= 50 and N % K == 0
# gabarito: o cliente c pertence ao grupo c * K // N (blocos contíguos de N/K clientes)
VERDADE = [c * K // N for c in range(N)]
print("GPU:", torch.cuda.is_available(), "| clientes por rodada:", {f: int(N * f) for f in FRACOES})

## 1. Execução

`executar(frac, seed)` roda o script uma vez. Se já existir uma execução concluída com os mesmos parâmetros (com `ROUNDS` entradas em `cluster.pkl`), ela é reaproveitada, a menos que `REEXECUTAR = True`.

Atenção: o script grava `psi.pkl` a cada rodada (`N` vetores de ~6,5 MB, ou seja, ~1,3 GB por gravação), então cada execução leva alguns minutos e usa espaço em `logs/`.

In [ ]:
def dados_prontos(seed):
    # as partições rotacionadas já foram gravadas? (decide o --process do script)
    d = PROJETO / f"datasets/rotated_mnist_{N}_{seed}"
    return len(list((d / "train").glob("data*.pkl"))) == N and len(list((d / "test").glob("data*.pkl"))) == K


def pastas_de_log(m, seed):
    # o nome da pasta de logs do script traz n, m, seed e mu, mas não o tau; ordena da mais antiga para a mais nova
    padrao = f"runs-rotated-mnist-n{N}-p{m}-seed{seed}-lambda{MU}-time-*"
    return sorted((PROJETO / "logs").glob(padrao), key=lambda p: p.stat().st_mtime)


def concluida(pasta):
    # execução completa = cluster.pkl tem uma partição por rodada
    try:
        return len(torch.load(pasta / "cluster.pkl", weights_only=False)) == ROUNDS
    except Exception:
        return False


def executar(frac, seed):
    m = int(N * frac)
    if not REEXECUTAR:   # reaproveita a execução mais recente que terminou
        for pasta in reversed(pastas_de_log(m, seed)):
            if concluida(pasta):
                print(f"m={frac:.0%} seed={seed}: reaproveitando {pasta.name}")
                return pasta, None
    cmd = [sys.executable, SCRIPT, "--n", str(N), "--com_round", str(ROUNDS),
           "--num_per_round", str(m), "--mu", str(MU), "--tau", str(TAU),
           "--train", "0", "--seed", str(seed), "--dataset", "mnist",
           "--process", "0" if dados_prontos(seed) else "1"]
    (PROJETO / "logs").mkdir(exist_ok=True)
    saida = PROJETO / "logs" / f"pipeline-m{m}-seed{seed}.txt"
    inicio = time.time()
    with open(saida, "w") as f:
        r = subprocess.run(cmd, cwd=PROJETO, stdout=f, stderr=subprocess.STDOUT)
    duracao = time.time() - inicio
    if r.returncode != 0:
        print(saida.read_text()[-3000:])
        raise RuntimeError(f"falhou: {' '.join(cmd)}")
    # a pasta criada por esta execução é a mais nova com mtime a partir do início
    pasta = next(p for p in reversed(pastas_de_log(m, seed)) if p.stat().st_mtime >= inicio - 1)
    print(f"m={frac:.0%} seed={seed}: {duracao:.0f} s -> {pasta.name}")
    return pasta, duracao


execucoes = {}
for seed in SEEDS:
    for frac in FRACOES:
        execucoes[(frac, seed)] = executar(frac, seed)

## 2. Métricas por rodada

O `cluster.pkl` guarda, para cada rodada, o dicionário `cluster_id → membros` (clusters mortos ficam com lista vazia). A cobertura vem das linhas `client id this round` do log principal.

In [ ]:
def rotulos(particao):
    # dicionário cluster -> membros vira lista cliente -> cluster (formato do adjusted_rand_score)
    r = [None] * N
    for cid, membros in particao.items():
        for c in membros:
            r[c] = cid
    return r


def pureza(particao):
    # fração de clientes que pertencem ao grupo real dominante do seu cluster
    ok = 0
    for membros in particao.values():
        if membros:
            ok += np.bincount([VERDADE[c] for c in membros]).max()
    return ok / N


def misturados(particao):
    # nº de clusters com clientes de mais de um grupo real (fusões erradas, irreversíveis)
    return sum(1 for membros in particao.values() if len({VERDADE[c] for c in membros}) > 1)


def cobertura(pasta):
    # fração acumulada de clientes já sorteados por rodada, lida das linhas "client id this round" do log
    visto, serie = set(), []
    for linha in (pasta / "rotated_mnist.log").read_text().splitlines():
        achou = re.search(r"client id this round \[(.*?)\]", linha)
        if achou:
            visto |= {int(x) for x in achou.group(1).split(",")}
            serie.append(len(visto) / N)
    return serie


def metricas(pasta):
    # uma linha por rodada; clusters mortos (lista vazia) não contam
    log = torch.load(pasta / "cluster.pkl", weights_only=False)
    cob = cobertura(pasta)
    linhas = []
    for r, particao in enumerate(log):
        linhas.append({
            "rodada": r + 1,
            "clusters": sum(1 for v in particao.values() if v),
            "ari": adjusted_rand_score(VERDADE, rotulos(particao)),
            "pureza": pureza(particao),
            "misturados": misturados(particao),
            "cobertura": cob[r],
        })
    return pd.DataFrame(linhas), log


def rodada_estavel(log):
    """Primeira rodada (1-based) a partir da qual a partição não muda mais."""
    chave = lambda p: frozenset(frozenset(v) for v in p.values() if v)
    final = chave(log[-1])
    r = len(log)
    while r > 1 and chave(log[r - 2]) == final:
        r -= 1
    return r


# resumo da última rodada de cada execução
tabelas, resumo = {}, []
for (frac, seed), (pasta, duracao) in execucoes.items():
    df, log = metricas(pasta)
    df.insert(0, "seed", seed)
    df.insert(0, "m", frac)
    tabelas[(frac, seed)] = df
    ultimo = df.iloc[-1]
    atingiu = df.index[df["clusters"] == K]   # rodadas em que o nº de clusters chegou a k
    resumo.append({
        "m": f"{frac:.0%}", "clientes/rodada": int(N * frac), "seed": seed,
        "clusters finais": int(ultimo["clusters"]),
        "1ª rodada com k=4": int(df.loc[atingiu[0], "rodada"]) if len(atingiu) else None,
        "rodada estável": rodada_estavel(log),
        "ARI final": round(ultimo["ari"], 3),
        "pureza final": round(ultimo["pureza"], 3),
        "clusters mistos": int(ultimo["misturados"]),
        "cobertura final": round(ultimo["cobertura"], 2),
        "tempo (s)": None if duracao is None else round(duracao),
    })

resumo = pd.DataFrame(resumo).astype({"1ª rodada com k=4": "Int64", "tempo (s)": "Int64"})
resumo

## 3. Gráficos

Gráficos interativos em Plotly: passe o mouse para ver os valores por rodada e clique na legenda para ocultar um valor de `m`. Um painel por métrica. Para mais de uma seed, a linha é a média e a faixa vai do mínimo ao máximo.

In [ ]:
CORES = {1.0: "#2a78d6", 0.5: "#eb6834", 0.25: "#1baf7a", 0.1: "#eda100"}
TRACOS = {1.0: ("solid", "circle"), 0.5: ("dash", "square"), 0.25: ("dashdot", "triangle-up"), 0.1: ("dot", "diamond")}

# um painel por métrica, uma linha por m
todas = pd.concat(tabelas.values(), ignore_index=True)
paineis = [("clusters", "Nº de clusters"), ("ari", "ARI contra os grupos reais"), ("cobertura", "Clientes já observados")]

fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in paineis], horizontal_spacing=0.07)
for i, (col, _) in enumerate(paineis, start=1):
    for frac in FRACOES:
        g = todas[todas["m"] == frac].groupby("rodada")[col]
        media = g.mean()
        dash, simbolo = TRACOS.get(frac, ("solid", "circle"))
        nome = f"m = {frac:.0%}"
        if len(SEEDS) > 1:  # faixa mín-máx entre as seeds
            fig.add_trace(go.Scatter(x=list(media.index) + list(media.index[::-1]),
                                     y=list(g.max()) + list(g.min()[::-1]), fill="toself",
                                     fillcolor=CORES.get(frac), opacity=0.15, line=dict(width=0),
                                     hoverinfo="skip", showlegend=False, legendgroup=nome), row=1, col=i)
        fig.add_trace(go.Scatter(x=media.index, y=media.values, mode="lines+markers", name=nome,
                                 legendgroup=nome, showlegend=(i == 1),
                                 line=dict(color=CORES.get(frac), dash=dash, width=2),
                                 marker=dict(symbol=simbolo, size=6, color=CORES.get(frac)),
                                 hovertemplate="rodada %{x}: %{y:.3g}<extra>" + nome + "</extra>"), row=1, col=i)
    fig.update_xaxes(title_text="rodada", row=1, col=i)
fig.add_hline(y=K, line=dict(color="gray", width=1), annotation_text=f"k real = {K}", annotation_position="top right", row=1, col=1)
fig.update_layout(height=420, hovermode="x unified", margin=dict(t=50, b=40, l=50, r=20),
                  legend=dict(orientation="h", y=-0.2), template="simple_white")
fig.update_xaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
fig.show()

## 4. Leitura

Rode as células acima e compare a tabela do item 2:

- **`rodada estável`** menor = a partição para de mudar mais cedo.
- **`clusters finais` ≠ 4** com `ARI` baixo indica que `τ` (`TAU`, 0.9 por padrão) não separou os grupos naquele `m`, ou que a clusterização não terminou nas 50 rodadas.
- **`cobertura`** mostra quanto da lentidão vem de clientes ainda não observados (o cliente só entra na fusão depois de sorteado).
- Com `m = 100%` todos são observados na rodada 1 e todos os clusters entram na matriz de cosseno a cada rodada, então é o limite mais rápido que o algoritmo atual permite.
- Com uma só seed, as diferenças pequenas entre os `m` podem ser ruído da amostragem. Para conclusões mais firmes, use `SEEDS = [0, 42, 1998]` (cada seed gera e grava uma nova partição dos dados).

## 5. Busca de τ

Como os clientes se agrupam em função de `τ`? O gabarito é o cenário ideal: os `k = 4` grupos reais (`VERDADE`). Para cada `m` em `TAU_FRACOES`:

1. **Varredura grossa** em `TAU_GRADE` (de 0.05 a 1.0). Com `τ = 1.0` nenhum par se funde (cosseno > 1 é impossível), então é o extremo "cada cliente é um cluster".
2. **Bisseção:** o melhor resultado é o `τ` de maior ARI final contra o gabarito (se vários empatam, o intervalo `[τ_min, τ_max]` deles). A cada passo, avalia-se o ponto médio do maior intervalo ainda não resolvido entre a borda desse conjunto ótimo e o vizinho avaliado. Para depois de `TAU_BISSECCOES` passos ou quando o intervalo fica menor que `TAU_LARGURA_MIN`.
3. **τ sugerido:** o centro do intervalo ótimo, que fica mais longe das duas bordas onde a partição piora.

Cada execução é uma chamada do script com `--tau` diferente; o resultado vai para `logs/busca-tau.json`, então reexecutar a célula reaproveita o que já foi rodado (a menos que `REEXECUTAR = True`). Só a primeira seed (`SEEDS[0]`) é usada. Atenção ao espaço em disco: cada execução grava ~1,3 GB em `logs/`.

In [ ]:
import json

# Parâmetros da busca de τ
TAU_GRADE = [0.05, 0.25, 0.5, 0.75, 1.0]   # varredura grossa, com os extremos 0.05 e 1.0
TAU_FRACOES = [1.0, 0.25, 0.1]             # m: uma busca separada para cada fração
TAU_BISSECCOES = 6                         # máximo de pontos avaliados depois da varredura
TAU_LARGURA_MIN = 0.005                    # para quando o maior intervalo aberto fica menor que isso
TAU_SEED = SEEDS[0]                        # só uma seed na busca
# o nome da pasta de logs não traz o tau, então guardamos aqui qual pasta é de qual (m, seed, tau)
CACHE_TAU = PROJETO / "logs" / "busca-tau.json"


def carregar_cache():
    try:
        return json.loads(CACHE_TAU.read_text())
    except Exception:
        return {}


def executar_tau(frac, seed, tau):
    # igual a executar(), mas com --tau variável e cache próprio
    m = int(N * frac)
    chave = f"{m}|{seed}|{tau:.4f}"
    cache = carregar_cache()
    if chave in cache and not REEXECUTAR and concluida(PROJETO / "logs" / cache[chave]):
        return PROJETO / "logs" / cache[chave]
    # o script nomeia a pasta só até o minuto: espera virar o minuto para não cair na pasta de outra execução
    while any(p.name.endswith(time.strftime("%m-%d-%H:%M")) for p in pastas_de_log(m, seed)):
        time.sleep(5)
    cmd = [sys.executable, SCRIPT, "--n", str(N), "--com_round", str(ROUNDS),
           "--num_per_round", str(m), "--mu", str(MU), "--tau", str(tau),
           "--train", "0", "--seed", str(seed), "--dataset", "mnist",
           "--process", "0" if dados_prontos(seed) else "1"]
    saida = PROJETO / "logs" / f"pipeline-m{m}-seed{seed}-tau{tau:.4f}.txt"
    inicio = time.time()
    with open(saida, "w") as f:
        r = subprocess.run(cmd, cwd=PROJETO, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(saida.read_text()[-3000:])
        raise RuntimeError(f"falhou: {' '.join(cmd)}")
    pasta = next(p for p in reversed(pastas_de_log(m, seed)) if p.stat().st_mtime >= inicio - 1)
    cache = carregar_cache()
    cache[chave] = pasta.name
    CACHE_TAU.write_text(json.dumps(cache, indent=1))
    return pasta


def avaliar_tau(frac, tau):
    # métricas da última rodada: é a partição final que importa na busca
    pasta = executar_tau(frac, TAU_SEED, tau)
    df, log = metricas(pasta)
    u = df.iloc[-1]
    return {"m": frac, "tau": tau, "clusters": int(u["clusters"]), "ari": float(u["ari"]),
            "pureza": float(u["pureza"]), "misturados": int(u["misturados"]),
            "rodada estável": rodada_estavel(log), "pasta": pasta.name}


def buscar_tau(frac):
    # varredura grossa e depois bisseção em torno do(s) τ de maior ARI contra o gabarito
    res = {}   # tau -> métricas

    def ver(tau, fase):
        # avalia o tau (se ainda não foi avaliado) e imprime o resultado
        tau = round(tau, 4)
        if tau not in res:
            res[tau] = {**avaliar_tau(frac, tau), "fase": fase}
            r = res[tau]
            print(f"m={frac:.0%} τ={tau:<7} {fase:<9} clusters={r['clusters']:<4} ARI={r['ari']:.3f} pureza={r['pureza']:.3f}")

    for t in TAU_GRADE:
        ver(t, "grade")
    for _ in range(TAU_BISSECCOES):
        taus = sorted(res)
        melhor = max(r["ari"] for r in res.values())
        otimos = [t for t in taus if res[t]["ari"] >= melhor - 1e-9]   # empates em ARI formam um intervalo
        i_lo, i_hi = taus.index(otimos[0]), taus.index(otimos[-1])
        abertos = []   # intervalos entre a borda do conjunto ótimo e o vizinho avaliado
        if i_lo > 0:
            abertos.append((taus[i_lo - 1], taus[i_lo]))
        if i_hi < len(taus) - 1:
            abertos.append((taus[i_hi], taus[i_hi + 1]))
        if not abertos:
            break
        a, b = max(abertos, key=lambda x: x[1] - x[0])   # refina primeiro o maior intervalo, onde a borda é mais incerta
        if b - a < TAU_LARGURA_MIN:
            break
        ver((a + b) / 2, "bisseção")
    return pd.DataFrame(res.values()).sort_values("tau").reset_index(drop=True)


# resultado: um DataFrame por m, ordenado por tau
busca = {frac: buscar_tau(frac) for frac in TAU_FRACOES}

### Resultados da busca

Para cada `m`: o intervalo de `τ` com o maior ARI, o `τ` sugerido (centro do intervalo) e a incerteza das bordas (o intervalo até o vizinho avaliado, onde a borda real está).

In [ ]:
# Para cada m: intervalo de τ com o maior ARI, τ sugerido (centro) e incerteza das bordas
linhas = []
for frac, d in busca.items():
    melhor = d["ari"].max()
    otimos = d[d["ari"] >= melhor - 1e-9]
    lo, hi = otimos["tau"].min(), otimos["tau"].max()
    taus = list(d["tau"])
    # vizinhos avaliados fora do intervalo ótimo: a borda real está entre eles e a borda do ótimo
    viz_lo = taus[taus.index(lo) - 1] if taus.index(lo) > 0 else None
    viz_hi = taus[taus.index(hi) + 1] if taus.index(hi) < len(taus) - 1 else None
    linhas.append({
        "m": f"{frac:.0%}", "execuções": len(d), "ARI máx": round(melhor, 3),
        "clusters no ótimo": sorted(set(otimos["clusters"])),
        "τ ótimo (mín)": lo, "τ ótimo (máx)": hi, "τ sugerido": round((lo + hi) / 2, 4),
        "borda inferior em": None if viz_lo is None else f"({viz_lo}, {lo}]",
        "borda superior em": None if viz_hi is None else f"[{hi}, {viz_hi})",
    })
display(pd.DataFrame(linhas))
# todas as execuções de cada m, em ordem de tau
for frac, d in busca.items():
    print(f"\nm = {frac:.0%}")
    display(d.drop(columns=["m", "pasta"]).round(3))

In [ ]:
# mesmos 3 painéis do item 3, mas em função de τ e não da rodada; losango = ponto da bisseção
paineis = [("clusters", "Nº de clusters finais"), ("ari", "ARI final contra os grupos reais"), ("pureza", "Pureza final")]
fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in paineis], horizontal_spacing=0.07)
for i, (col, _) in enumerate(paineis, start=1):
    for frac, d in busca.items():
        dash, _ = TRACOS.get(frac, ("solid", "circle"))
        nome = f"m = {frac:.0%}"
        fig.add_trace(go.Scatter(
            x=d["tau"], y=d[col], mode="lines+markers", name=nome, legendgroup=nome, showlegend=(i == 1),
            line=dict(color=CORES.get(frac), dash=dash, width=2),
            marker=dict(symbol=[("diamond" if f == "bisseção" else "circle") for f in d["fase"]], size=8,
                        color=CORES.get(frac)),
            hovertemplate="τ=%{x}: %{y:.3g}<extra>" + nome + "</extra>"), row=1, col=i)
    fig.update_xaxes(title_text="τ (losango = bisseção)", row=1, col=i)
# linha de referência: k real
fig.add_hline(y=K, line=dict(color="gray", width=1), annotation_text=f"k real = {K}", annotation_position="top right", row=1, col=1)
fig.update_layout(height=420, hovermode="x unified", margin=dict(t=50, b=40, l=50, r=20),
                  legend=dict(orientation="h", y=-0.2), template="simple_white")
fig.update_xaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
fig.show()